# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuratulainAzhar22/flyrank-ml-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



I chose a Decision Tree Classifier for this modeling lane.

The Week-4 baseline is a transparent rule based on current search visibility, CTR, average position, and impression volume. It is useful as a simple decision-support baseline, but it does not learn relationships between multiple signals.

A Decision Tree is a suitable next step because it can learn non-linear relationships between the available performance and engagement features while remaining relatively easy to inspect and explain. I will keep the tree shallow rather than optimizing only for a higher score, because the goal is a useful and interpretable model rather than unnecessary complexity.

The model will be evaluated against the Week-4 baseline using the same target and an honest client-grouped split.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



I use a grouped train/test split by `client_hash_id`.

The reason is that multiple rows can belong to the same client. If rows from the same client appeared in both training and testing data, the model could benefit from client-specific patterns and produce an overly optimistic evaluation.

I therefore keep every client entirely in either the training or test set. The test set contains clients that were not used for training.

I use an 80/20 grouped split with a fixed random seed so that the experiment is reproducible.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("My_Read_Token")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection configured.")

Hugging Face connection configured.


In [2]:
march_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT *
FROM read_parquet('{march_path}')
""")

print("March 2026 view created.")

March 2026 view created.


In [3]:
check = con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS clients
FROM march
""").df()

check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,min_date,max_date,clients
0,9841378,2026-03-01,2026-03-31,55


In [4]:
model_query = """
WITH base AS (

    SELECT
        client_hash_id,

        ga4_pageviews,
        ga4_sessions,
        ga4_users,
        ga4_engaged_sessions,
        ga4_total_engagement_sec,

        sessions_organic,
        sessions_direct,
        sessions_referral,
        sessions_social,
        sessions_paid,
        sessions_ai,

        scroll_events,

        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_avg_position < 4 THEN '1-3'
            WHEN gsc_avg_position < 11 THEN '4-10'
            WHEN gsc_avg_position < 21 THEN '11-20'
            ELSE '21+'
        END AS position_bucket,

        gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr

    FROM march

    WHERE
        gsc_data_available IS TRUE
        AND gsc_impressions > 0
        AND gsc_avg_position IS NOT NULL
),

benchmarks AS (

    SELECT
        position_bucket,
        SUM(gsc_clicks) * 1.0 /
        NULLIF(SUM(gsc_impressions), 0) AS benchmark_ctr
    FROM base
    GROUP BY position_bucket
)

SELECT
    b.*,
    p.benchmark_ctr,

    CASE
        WHEN b.gsc_impressions >= 100
             AND b.ctr < p.benchmark_ctr
        THEN 1

        WHEN b.gsc_impressions < 100
             AND b.ctr < p.benchmark_ctr
        THEN 1

        ELSE 0
    END AS target

FROM base b

LEFT JOIN benchmarks p
    ON b.position_bucket = p.position_bucket
"""

model_data = con.sql(model_query)

print("Model dataset prepared.")

Model dataset prepared.


In [7]:
sample = con.sql("""
SELECT *
FROM model_data
USING SAMPLE reservoir(500000 ROWS)
REPEATABLE (42)
""").df()

print("Rows:", len(sample))
print("Clients:", sample["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 500000
Clients: 43


In [8]:
feature_cols = [
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

X = sample[feature_cols].copy()
y = sample["target"].copy()
groups = sample["client_hash_id"].copy()

print("X shape:", X.shape)
print("Target shape:", y.shape)
print("Unique clients:", groups.nunique())

X shape: (500000, 12)
Target shape: (500000,)
Unique clients: 43


In [9]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("Missing values:", X_imputed.isna().sum().sum())

Missing values: 0


In [10]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X_imputed,
        y,
        groups=groups
    )
)

X_train = X_imputed.iloc[train_idx]
X_test = X_imputed.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = groups.iloc[train_idx]
test_clients = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", train_clients.nunique())
print("Test clients:", test_clients.nunique())

print(
    "Client overlap:",
    len(set(train_clients) & set(test_clients))
)

Training rows: 455641
Test rows: 44359
Training clients: 34
Test clients: 9
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Week-4 baseline is the transparent CTR-versus-position and impression-volume rule. It prioritizes rows when CTR is below the benchmark for the row's position bucket, with impression volume used to distinguish stronger and weaker opportunities.

For Week 5, I train a Decision Tree using the selected numeric features and evaluate it on the same client-grouped test set.

I use Macro F1 as the primary classification metric because the two target classes should both be considered rather than allowing the majority class to dominate the evaluation.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 3. Train + compare vs my baseline
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42,
    class_weight="balanced"
)

dt_model.fit(X_train, y_train)

print("Decision Tree trained successfully.")


Decision Tree trained successfully.


In [12]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

y_pred = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, y_pred)
dt_f1 = f1_score(y_test, y_pred, average="macro")
dt_precision = precision_score(y_test, y_pred, average="macro")
dt_recall = recall_score(y_test, y_pred, average="macro")

print("Decision Tree Accuracy:", round(dt_accuracy, 4))
print("Decision Tree Macro F1:", round(dt_f1, 4))
print("Decision Tree Macro Precision:", round(dt_precision, 4))
print("Decision Tree Macro Recall:", round(dt_recall, 4))

Decision Tree Accuracy: 0.8743
Decision Tree Macro F1: 0.5974
Decision Tree Macro Precision: 0.6879
Decision Tree Macro Recall: 0.5769


**Compare again the week 4 baseline**

In [13]:
baseline_pred = (
    (
        sample.iloc[test_idx]["gsc_impressions"].values >= 100
    )
    &
    (
        sample.iloc[test_idx]["ctr"].values
        <
        sample.iloc[test_idx]["benchmark_ctr"].values
    )
).astype(int)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    average="macro"
)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

print("Week-4 Baseline Accuracy:", round(baseline_accuracy, 4))
print("Week-4 Baseline Macro F1:", round(baseline_f1, 4))

Week-4 Baseline Accuracy: 0.1917
Week-4 Baseline Macro F1: 0.1891


**Comparison Table**

In [14]:
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Decision Tree"
    ],
    "Accuracy": [
        baseline_accuracy,
        dt_accuracy
    ],
    "Macro F1": [
        baseline_f1,
        dt_f1
    ]
})

comparison

,Method,Accuracy,Macro F1
0,Week-4 Baseline,0.191686,0.189096
1,Decision Tree,0.874253,0.597396


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


The Decision Tree is evaluated on clients that were not present in training. I focus on the errors rather than treating the highest metric as the only objective.

I inspect false positives and false negatives to understand where the model disagrees with the target. I also inspect feature importance to see which available signals the tree relies on.

The results should be interpreted as measured patterns in this dataset rather than proof of causation. The model is decision-support, not a guarantee that a particular content item needs a specific intervention.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[  998  4507]
 [ 1071 37783]]


In [16]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": dt_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance

,feature,importance
5,sessions_organic,0.964808
0,ga4_pageviews,0.022207
1,ga4_sessions,0.006193
11,scroll_events,0.003519
2,ga4_users,0.002575
10,sessions_ai,0.000258
4,ga4_total_engagement_sec,0.000203
6,sessions_direct,0.000160
9,sessions_paid,0.000078
3,ga4_engaged_sessions,0.000000


### Feature interpretation

The feature-importance output shows which variables the Decision Tree used most heavily when making its splits.

The most important features are useful for understanding the model's decision process, but feature importance does not establish causation. A high importance value means that the feature contributed strongly to the model's predictions in this experiment; it does not mean that changing that feature would necessarily cause the target outcome to change.

The grouped client split also means that the reported performance reflects generalization to unseen clients rather than memorization of the same clients used during training.

In [17]:
errors = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred
})

errors["error_type"] = "correct"

errors.loc[
    (errors["actual"] == 0) & (errors["predicted"] == 1),
    "error_type"
] = "false_positive"

errors.loc[
    (errors["actual"] == 1) & (errors["predicted"] == 0),
    "error_type"
] = "false_negative"

print(errors["error_type"].value_counts())

error_type
correct           38781
false_positive     4507
false_negative     1071
Name: count, dtype: int64


### Error analysis

The error counts show the balance between false positives and false negatives.

False positives are cases where the model predicts the review/opportunity class but the target is 0. False negatives are cases where the target is 1 but the model predicts 0.

These errors matter because the model is intended for prioritization. A false positive may cause a reviewer to spend time investigating an item that does not represent a strong opportunity. A false negative may cause a potentially useful item to be missed.

The model should therefore be treated as decision-support rather than an automatic action system.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.